# Cartwheel Galaxy: Collision & Starburst Ring — Educational YouTube Shorts Notebook

This notebook generates a **15+ second vertical Shorts video** for students. The style is synthetic telescope/space visualization with HUD overlays, learning points, and a final reveal.

**Teaching hook:** A galaxy collision turned gravity into a cosmic ripple.

## Learning points
- The Cartwheel Galaxy’s ring shape was caused by a galactic collision.
- The expanding ring compresses gas and triggers star formation.
- Spokes and arcs show how galaxies are reshaped by gravity.
- Collisions can create new structure instead of simply destroying galaxies.

**Output:** 720×1280, 16.5 seconds, MP4 + preview PNG.

Run the cells from top to bottom.


In [ ]:
# Uncomment in a fresh environment if needed:
# %pip install -U numpy pillow imageio imageio-ffmpeg


In [1]:

from __future__ import annotations

from pathlib import Path
import math
import numpy as np
from PIL import Image, ImageDraw, ImageFont, ImageFilter
import imageio.v2 as imageio

# =============================
# YouTube Shorts video settings
# =============================
OUT_DIR = Path('.')
W, H = 720, 1280          # vertical 9:16 Shorts format
FPS = 18
SECONDS = 16.5            # at least 15 seconds
NFRAMES = int(FPS * SECONDS)

# Change TARGET to render a different object. In individual notebooks this is already set.
TARGET = 'cartwheel'
RENDER_ALL = False

PRESETS = {'mars': {'display_name': 'Mars: Volcanoes, Canyons & Ancient Water', 'short_name': 'Mars', 'kind': 'planet_mars', 'hook': 'Why Mars is the best classroom for planetary geology.', 'facts': ['Olympus Mons is the largest volcano in the Solar System.', 'Valles Marineris is a canyon system thousands of kilometers long.', 'Red iron-rich dust gives Mars its familiar color.', 'Rovers study rocks for clues about ancient water.'], 'keywords': ['geology', 'ancient water', 'volcano', 'canyon'], 'reveal': 'MARS: A frozen desert with giant scars', 'filename': 'Mars_Geology_Educational_Shorts.ipynb', 'description': 'A student-friendly Mars Shorts animation showing why Mars is a key world for learning planetary geology, volcanoes, canyons, dust, and ancient water.'}, 'jupiter': {'display_name': 'Jupiter: The Great Red Spot & Gas Giants', 'short_name': 'Jupiter', 'kind': 'planet_jupiter', 'hook': 'A giant planet where storms can outlive civilizations.', 'facts': ['Jupiter is the largest planet in our Solar System.', 'The Great Red Spot is a long-lived giant storm.', 'Jupiter rotates so fast that one day is about 10 hours.', 'Its gravity shaped the architecture of the outer Solar System.'], 'keywords': ['gas giant', 'storm', 'rotation', 'gravity'], 'reveal': 'JUPITER: Storm laboratory of the Solar System', 'filename': 'Jupiter_Great_Red_Spot_Educational_Shorts.ipynb', 'description': 'A Jupiter educational Shorts notebook focused on the Great Red Spot, gas giants, rapid rotation, and why Jupiter matters in Solar System science.'}, 'saturn': {'display_name': 'Saturn: Rings, Moons & Gravity', 'short_name': 'Saturn', 'kind': 'planet_saturn', 'hook': 'Saturn turns gravity into a visible physics lesson.', 'facts': ['Saturn’s rings are mostly ice particles mixed with rock and dust.', 'The rings are wide but extremely thin compared with the planet.', 'Moons sculpt gaps and waves inside the rings.', 'Titan shows students that moons can have complex atmospheres.'], 'keywords': ['rings', 'gravity', 'moons', 'resonance'], 'reveal': 'SATURN: Rings that reveal gravity in action', 'filename': 'Saturn_Rings_Educational_Shorts.ipynb', 'description': 'A Saturn educational Shorts notebook explaining rings, moon gravity, resonance, icy particles, and why Saturn is perfect for teaching orbital physics.'}, 'venus': {'display_name': 'Venus: Greenhouse Effect & Climate Extremes', 'short_name': 'Venus', 'kind': 'planet_venus', 'hook': 'Venus is Earth’s warning label in the sky.', 'facts': ['Venus has the hottest planetary surface in the Solar System.', 'Its thick atmosphere is mostly carbon dioxide.', 'Sulfuric-acid clouds hide the surface from visible light.', 'Venus teaches how atmosphere controls planetary climate.'], 'keywords': ['greenhouse', 'CO2', 'atmosphere', 'climate'], 'reveal': 'VENUS: Climate science at planetary scale', 'filename': 'Venus_Greenhouse_Effect_Educational_Shorts.ipynb', 'description': 'A Venus educational Shorts notebook focused on greenhouse effect, carbon dioxide, atmospheric pressure, clouds, and planetary climate lessons.'}, 'neptune': {'display_name': 'Neptune: Fast Winds & Ice Giant Science', 'short_name': 'Neptune', 'kind': 'planet_neptune', 'hook': 'The farthest major planet still has some of the fastest winds.', 'facts': ['Neptune is the windiest planet in the Solar System.', 'Frozen methane clouds race through its atmosphere.', 'Its blue color comes mainly from methane absorption.', 'Ice giants help students compare planet interiors.'], 'keywords': ['ice giant', 'winds', 'methane', 'outer Solar System'], 'reveal': 'NEPTUNE: The distant world with extreme winds', 'filename': 'Neptune_Fast_Winds_Educational_Shorts.ipynb', 'description': 'A Neptune educational Shorts notebook showing ice giant science, methane clouds, blue color, and extreme winds in the outer Solar System.'}, 'earth_moon': {'display_name': 'Earth–Moon System: Tides, Phases & Eclipses', 'short_name': 'Earth–Moon', 'kind': 'earth_moon', 'hook': 'The Moon is not just a night light — it shapes Earth systems.', 'facts': ['The Moon’s gravity drives Earth’s ocean tides.', 'Moon phases come from changing viewing angles of sunlight.', 'Eclipses happen only when the Sun, Earth, and Moon align.', 'The Moon preserves impact history in its ancient craters.'], 'keywords': ['tides', 'phases', 'eclipses', 'gravity'], 'reveal': 'EARTH–MOON: Gravity you can see every day', 'filename': 'Earth_Moon_Tides_Eclipses_Educational_Shorts.ipynb', 'description': 'An Earth–Moon educational Shorts notebook covering tides, moon phases, eclipses, gravity, and why our Moon matters to Earth.'}, 'milkyway_center': {'display_name': 'Milky Way Center: Sagittarius A*', 'short_name': 'Milky Way Center', 'kind': 'galactic_center', 'hook': 'At the center of our galaxy, stars orbit an invisible heavyweight.', 'facts': ['Sagittarius A* is the supermassive black hole at the Milky Way’s center.', 'Its mass is about four million Suns.', 'Star orbits near the center reveal the hidden black hole.', 'Infrared observations help astronomers see through galactic dust.'], 'keywords': ['black hole', 'star orbits', 'gravity', 'infrared'], 'reveal': 'MILKY WAY CENTER: Stars orbit the invisible', 'filename': 'Milky_Way_Center_Sagittarius_A_Educational_Shorts.ipynb', 'description': 'A Milky Way center educational Shorts notebook about Sagittarius A*, stellar orbits, infrared astronomy, dust, and black hole gravity.'}, 'm87': {'display_name': 'M87: First Imaged Black Hole & Jet', 'short_name': 'M87', 'kind': 'm87_blackhole', 'hook': 'The galaxy that gave humanity its first black-hole image.', 'facts': ['M87 contains the first black hole ever directly imaged by the EHT.', 'The bright ring is hot gas bent by extreme gravity.', 'A powerful jet launches from the galaxy’s central engine.', 'Black holes can influence the evolution of their host galaxies.'], 'keywords': ['black hole', 'EHT', 'jet', 'relativity'], 'reveal': 'M87: Where humanity saw a black hole', 'filename': 'M87_Black_Hole_Jet_Educational_Shorts.ipynb', 'description': 'An M87 educational Shorts notebook about the first black hole image, accretion light, relativistic jets, and galaxy feedback.'}, 'lmc': {'display_name': 'Large Magellanic Cloud: Star Birth Nearby', 'short_name': 'Large Magellanic Cloud', 'kind': 'lmc_starbirth', 'hook': 'A nearby galaxy that works like a star-formation laboratory.', 'facts': ['The Large Magellanic Cloud is a satellite galaxy of the Milky Way.', 'The Tarantula Nebula is an extremely active star-forming region.', 'Massive young stars sculpt gas and dust with radiation and winds.', 'Nearby dwarf galaxies help students study galaxy evolution.'], 'keywords': ['dwarf galaxy', 'star birth', 'nebula', 'Local Group'], 'reveal': 'LMC: A nearby galaxy making new stars', 'filename': 'Large_Magellanic_Cloud_Star_Birth_Educational_Shorts.ipynb', 'description': 'A Large Magellanic Cloud educational Shorts notebook about star formation, the Tarantula Nebula, dwarf galaxies, and galaxy evolution.'}, 'cartwheel': {'display_name': 'Cartwheel Galaxy: Collision & Starburst Ring', 'short_name': 'Cartwheel Galaxy', 'kind': 'cartwheel_galaxy', 'hook': 'A galaxy collision turned gravity into a cosmic ripple.', 'facts': ['The Cartwheel Galaxy’s ring shape was caused by a galactic collision.', 'The expanding ring compresses gas and triggers star formation.', 'Spokes and arcs show how galaxies are reshaped by gravity.', 'Collisions can create new structure instead of simply destroying galaxies.'], 'keywords': ['collision', 'ring galaxy', 'starburst', 'gravity'], 'reveal': 'CARTWHEEL: A galaxy collision frozen in time', 'filename': 'Cartwheel_Galaxy_Collision_Educational_Shorts.ipynb', 'description': 'A Cartwheel Galaxy educational Shorts notebook explaining galactic collision, expanding rings, starburst regions, and gravitational reshaping.'}}

rng = np.random.default_rng(42)

try:
    FONT_BOLD = ImageFont.truetype('/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf', 32)
    FONT = ImageFont.truetype('/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf', 25)
    FONT_SMALL = ImageFont.truetype('/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf', 19)
    FONT_TINY = ImageFont.truetype('/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf', 15)
except Exception:
    FONT_BOLD = FONT = FONT_SMALL = FONT_TINY = ImageFont.load_default()


def np_to_img(arr):
    arr = np.clip(arr, 0, 255).astype(np.uint8)
    return Image.fromarray(arr, 'RGB')


def add_starfield(arr, seed=0, amount=1000):
    local = np.random.default_rng(seed)
    h, w = arr.shape[:2]
    ys = local.integers(0, h, amount)
    xs = local.integers(0, w, amount)
    brightness = local.integers(90, 255, amount)
    for x, y, b in zip(xs, ys, brightness):
        arr[y, x] = np.maximum(arr[y, x], b)
        if local.random() < 0.12 and 1 <= x < w-1 and 1 <= y < h-1:
            arr[y-1:y+2, x-1:x+2] = np.maximum(arr[y-1:y+2, x-1:x+2], b * 0.45)
    return arr


def background(seed=0):
    y = np.linspace(0, 1, H)[:, None]
    x = np.linspace(0, 1, W)[None, :]
    arr = np.zeros((H, W, 3), dtype=np.float32)
    arr[..., 0] = 5 + 12 * (1-y) + 2*np.sin(6*x)
    arr[..., 1] = 7 + 10 * (1-y)
    arr[..., 2] = 18 + 24 * (1-y)
    arr = add_starfield(arr, seed=seed, amount=1350)
    return arr


def draw_glow(arr, cx, cy, radius, color, strength=1.0, squash_y=1.0):
    yy, xx = np.indices((H, W))
    d2 = ((xx-cx)**2 + ((yy-cy)/squash_y)**2) / max(radius, 1)**2
    glow = np.exp(-d2 * 2.7) * 255 * strength
    for i, c in enumerate(color):
        arr[..., i] += glow * (c / 255.0)
    return arr


def add_text_box(draw, xy, title, lines, accent=(255,255,255)):
    x, y = xy
    max_w = W - 80
    pad = 18
    line_h = 26
    h = 54 + line_h * len(lines)
    draw.rounded_rectangle([x, y, x+max_w, y+h], radius=22, fill=(5, 10, 20, 178), outline=accent, width=2)
    draw.text((x+pad, y+12), title, fill=accent, font=FONT_BOLD)
    for i, line in enumerate(lines):
        draw.text((x+pad, y+48+i*line_h), line, fill=(235,240,255), font=FONT_SMALL)


def draw_hud(img, preset, frame_i, progress):
    img = img.convert('RGBA')
    overlay = Image.new('RGBA', img.size, (0,0,0,0))
    d = ImageDraw.Draw(overlay)
    accent = (110, 230, 255, 215)
    warm = (255, 218, 135, 225)

    # top header
    d.rounded_rectangle([30, 32, W-30, 130], radius=22, fill=(0, 12, 26, 160), outline=accent, width=2)
    d.text((52, 48), preset['display_name'].upper(), font=FONT_BOLD, fill=(245, 250, 255, 255))
    d.text((52, 88), preset['hook'], font=FONT_SMALL, fill=(190, 225, 255, 255))

    # scan sweep
    scan_y = int(155 + (H-250) * ((frame_i % 110) / 110))
    d.line([40, scan_y, W-40, scan_y], fill=(130, 255, 255, 120), width=3)
    d.rectangle([40, scan_y-18, W-40, scan_y], fill=(80, 210, 255, 25))

    # reticle and corner markers
    cx, cy = W//2, int(H*0.51)
    d.ellipse([cx-190, cy-190, cx+190, cy+190], outline=(120,230,255,120), width=2)
    d.line([cx-240, cy, cx-70, cy], fill=(120,230,255,120), width=2)
    d.line([cx+70, cy, cx+240, cy], fill=(120,230,255,120), width=2)
    d.line([cx, cy-240, cx, cy-70], fill=(120,230,255,120), width=2)
    d.line([cx, cy+70, cx, cy+240], fill=(120,230,255,120), width=2)

    # progress panel
    panel_y = H - 200
    d.rounded_rectangle([34, panel_y, W-34, H-38], radius=22, fill=(0, 12, 24, 165), outline=(255,255,255,80), width=1)
    d.text((54, panel_y+18), 'EDUCATIONAL LIVE VIEW', font=FONT_SMALL, fill=warm)
    d.text((54, panel_y+50), f'Signal clarity: {int(progress*100):03d}%   Frame stack: {frame_i+1:03d}/{NFRAMES}', font=FONT_TINY, fill=(225,240,255,255))
    d.rectangle([54, panel_y+82, W-54, panel_y+104], outline=(170,210,255,170), width=1)
    d.rectangle([56, panel_y+84, int(56+(W-112)*progress), panel_y+102], fill=(100,220,255,185))
    d.text((54, panel_y+118), 'Keywords: ' + '  •  '.join(preset['keywords']), font=FONT_TINY, fill=(235,240,255,240))

    # fact cards change across video
    facts = preset['facts']
    if progress < 0.28:
        title, lines = 'LEARNING POINT 1', [facts[0], facts[1]]
        add_text_box(d, (40, 180), title, lines, accent=(255,220,120,220))
    elif progress < 0.56:
        title, lines = 'LEARNING POINT 2', [facts[2]]
        add_text_box(d, (40, 180), title, lines, accent=(150,255,180,220))
    elif progress < 0.82:
        title, lines = 'WHY STUDENTS SHOULD CARE', [facts[3]]
        add_text_box(d, (40, 180), title, lines, accent=(220,180,255,220))
    else:
        d.rounded_rectangle([40, 178, W-40, 300], radius=26, fill=(0,0,0,175), outline=(255,255,255,180), width=2)
        d.text((64, 204), preset['reveal'], font=FONT_BOLD, fill=(255,255,255,255))
        d.text((64, 250), 'Pause, observe, ask: what physical process shaped this view?', font=FONT_SMALL, fill=(230,245,255,255))

    return Image.alpha_composite(img, overlay).convert('RGB')


def planet_mask(cx, cy, rx, ry):
    yy, xx = np.indices((H, W))
    m = ((xx-cx)/rx)**2 + ((yy-cy)/ry)**2 <= 1
    nx = (xx-cx)/rx
    ny = (yy-cy)/ry
    shade = np.clip(1.1 - 0.52*nx - 0.32*ny - 0.22*(nx**2+ny**2), 0, 1)
    return m, nx, ny, shade


def render_clean_scene(preset, t=0.85):
    kind = preset['kind']
    arr = background(seed=17)
    yy, xx = np.indices((H, W))
    cx, cy = W//2, int(H*0.53)

    if kind == 'planet_mars':
        rx, ry = 250, 250
        m, nx, ny, shade = planet_mask(cx, cy, rx, ry)
        base = np.zeros_like(arr)
        texture = 0.5 + 0.28*np.sin(8*nx + 5*np.sin(4*ny)) + 0.18*np.sin(17*ny + 2*np.cos(8*nx))
        base[...,0] = 190 + 60*texture
        base[...,1] = 78 + 35*texture
        base[...,2] = 45 + 20*texture
        # Valles-like dark canyon
        canyon = np.exp(-((ny + 0.08*np.sin(8*nx))**2)/0.01) * (np.abs(nx)<0.75)
        base[...,0] -= canyon*75; base[...,1] -= canyon*42; base[...,2] -= canyon*20
        # Polar cap
        cap = ((ny < -0.64) & m)
        base[cap] = base[cap]*0.35 + np.array([240,230,215])*0.65
        arr[m] = arr[m]*0.05 + base[m]*shade[m,None]
        arr = draw_glow(arr, cx, cy, 360, (255,100,55), 0.25)

    elif kind == 'planet_jupiter':
        rx, ry = 280, 260
        m, nx, ny, shade = planet_mask(cx, cy, rx, ry)
        bands = np.sin((ny*12 + 0.8*np.sin(nx*7))*math.pi)
        turbulence = np.sin(nx*23 + ny*8) * np.sin(ny*16)
        base = np.zeros_like(arr)
        base[...,0] = 190 + 30*bands + 18*turbulence
        base[...,1] = 150 + 22*bands + 9*turbulence
        base[...,2] = 95 + 12*bands
        # Great Red Spot
        spot = (((nx-0.35)/0.26)**2 + ((ny-0.08)/0.115)**2) < 1
        base[spot] = np.array([220, 92, 58])
        ring = ((((nx-0.35)/0.30)**2 + ((ny-0.08)/0.14)**2) < 1) & ~spot
        base[ring] = base[ring]*0.45 + np.array([255,205,150])*0.55
        arr[m] = arr[m]*0.05 + base[m]*shade[m,None]
        arr = draw_glow(arr, cx, cy, 400, (255,190,100), 0.18)

    elif kind == 'planet_saturn':
        # rings behind planet
        ring = (((xx-cx)/380)**2 + ((yy-cy)/82)**2 < 1) & (((xx-cx)/170)**2 + ((yy-cy)/36)**2 > 1)
        tilt = np.clip(1 - np.abs((yy-cy)/90), 0, 1)
        arr[ring] = arr[ring]*0.25 + np.array([210,185,130])*tilt[ring,None]*0.9
        rx, ry = 220, 210
        m, nx, ny, shade = planet_mask(cx, cy, rx, ry)
        bands = np.sin(ny*16)
        base = np.zeros_like(arr)
        base[...,0] = 208 + 20*bands
        base[...,1] = 174 + 14*bands
        base[...,2] = 108 + 9*bands
        arr[m] = arr[m]*0.05 + base[m]*shade[m,None]
        # rings in front of planet lower half
        front = ring & (yy > cy)
        arr[front] = arr[front]*0.25 + np.array([230,210,160])*tilt[front,None]*0.95
        arr = draw_glow(arr, cx, cy, 380, (255,220,140), 0.16)

    elif kind == 'planet_venus':
        rx, ry = 245, 245
        m, nx, ny, shade = planet_mask(cx, cy, rx, ry)
        swirl = np.sin(14*ny + 5*np.sin(6*nx)) + 0.45*np.sin(11*nx - 6*ny)
        base = np.zeros_like(arr)
        base[...,0] = 215 + 36*swirl
        base[...,1] = 160 + 22*swirl
        base[...,2] = 70 + 10*swirl
        arr[m] = arr[m]*0.05 + base[m]*shade[m,None]
        # bright cloud veil
        veil = m & (np.sin(9*nx + 13*ny)>0.55)
        arr[veil] = arr[veil]*0.5 + np.array([255,225,150])*0.5
        arr = draw_glow(arr, cx, cy, 390, (255,170,80), 0.3)

    elif kind == 'planet_neptune':
        rx, ry = 245, 245
        m, nx, ny, shade = planet_mask(cx, cy, rx, ry)
        bands = np.sin(10*ny + 2*np.sin(5*nx))
        base = np.zeros_like(arr)
        base[...,0] = 45 + 15*bands
        base[...,1] = 110 + 25*bands
        base[...,2] = 220 + 30*bands
        storm = (((nx+0.28)/0.25)**2 + ((ny-0.18)/0.12)**2) < 1
        base[storm] = np.array([25,55,145])
        cloud = (((nx-0.2)/0.5)**2 + ((ny+0.25)/0.06)**2) < 1
        base[cloud] = base[cloud]*0.25 + np.array([185,225,255])*0.75
        arr[m] = arr[m]*0.05 + base[m]*shade[m,None]
        arr = draw_glow(arr, cx, cy, 390, (65,130,255), 0.22)

    elif kind == 'earth_moon':
        # Earth
        ex, ey = W//2 - 80, int(H*0.54)
        rx, ry = 190, 190
        m, nx, ny, shade = planet_mask(ex, ey, rx, ry)
        land = (np.sin(6*nx) + np.sin(8*ny+2*nx) + np.sin(12*(nx+ny))) > 0.45
        base = np.zeros_like(arr)
        base[...,0] = np.where(land, 55, 24)
        base[...,1] = np.where(land, 150, 95)
        base[...,2] = np.where(land, 70, 200)
        clouds = m & (np.sin(18*nx + 7*ny) > 0.72)
        arr[m] = arr[m]*0.05 + base[m]*shade[m,None]
        arr[clouds] = arr[clouds]*0.55 + np.array([245,245,245])*0.45
        # Moon
        mx, my = W//2 + 195, int(H*0.43)
        mm, mnx, mny, msh = planet_mask(mx, my, 72, 72)
        moon = np.zeros_like(arr) + np.array([185,180,170])
        crater = (np.sin(18*mnx)*np.sin(17*mny) > 0.75) & mm
        moon[crater] = np.array([105,105,105])
        arr[mm] = arr[mm]*0.1 + moon[mm]*msh[mm,None]
        # tidal bulges
        arr = draw_glow(arr, ex-60, ey, 165, (80,160,255), 0.15, squash_y=0.65)
        arr = draw_glow(arr, ex+60, ey, 165, (80,160,255), 0.15, squash_y=0.65)

    elif kind == 'galactic_center':
        # dense star/dust field with hidden center
        arr = add_starfield(arr, seed=81, amount=5000)
        arr = draw_glow(arr, cx, cy, 430, (255,210,150), 0.45, squash_y=0.65)
        dust = np.exp(-((yy - (cy + 30*np.sin((xx-cx)/90)))**2)/(2*48**2))
        arr[...,0] -= dust*80; arr[...,1] -= dust*75; arr[...,2] -= dust*70
        arr = draw_glow(arr, cx, cy, 110, (255,255,210), 0.9)
        # orbit tracers
        img = np_to_img(arr).convert('RGBA')
        d = ImageDraw.Draw(img)
        for r, ang, col in [(150, 0.2, (160,220,255,160)), (220, 0.7, (255,200,150,130)), (90, 1.7, (180,255,180,120))]:
            box = [cx-r, cy-r//2, cx+r, cy+r//2]
            d.arc(box, start=15, end=335, fill=col, width=2)
            sx = cx + r*math.cos(ang)/1.0
            sy = cy + (r/2)*math.sin(ang)
            d.ellipse([sx-5, sy-5, sx+5, sy+5], fill=(255,255,255,220))
        arr = np.array(img.convert('RGB')).astype(np.float32)

    elif kind == 'm87_blackhole':
        arr = add_starfield(arr, seed=92, amount=2400)
        arr = draw_glow(arr, cx, cy, 420, (255,210,150), 0.3, squash_y=0.75)
        # jet
        for offset in range(-18, 19):
            line = np.abs((yy - (cy - 1.7*(xx-cx) + offset))) < 4
            line &= (xx > cx + 30)
            arr[line] = arr[line]*0.25 + np.array([130,210,255])*0.75
        # black hole ring
        r = np.sqrt((xx-cx)**2 + (yy-cy)**2)
        ring = np.exp(-((r-115)**2)/(2*14**2))
        asym = 0.55 + 0.65*np.clip((yy-cy)/130 + 0.2, 0, 1)
        arr[...,0] += ring*255*asym
        arr[...,1] += ring*160*asym
        arr[...,2] += ring*50*asym
        shadow = r < 78
        arr[shadow] *= 0.03
        arr = draw_glow(arr, cx, cy, 230, (255,140,55), 0.28)

    elif kind == 'lmc_starbirth':
        arr = add_starfield(arr, seed=64, amount=3500)
        # irregular galaxy patches
        centers = [(cx-110, cy+40, 230, (120,180,255), .28), (cx+90, cy-55, 190, (255,150,210), .22), (cx+130, cy+130, 160, (110,255,180), .18)]
        for gx, gy, gr, col, st in centers:
            arr = draw_glow(arr, gx, gy, gr, col, st, squash_y=0.72)
        # Tarantula nebula bright patch
        arr = draw_glow(arr, cx+95, cy-75, 130, (255,80,160), 0.85, squash_y=0.8)
        arr = draw_glow(arr, cx+95, cy-75, 80, (100,220,255), 0.55, squash_y=1.0)
        dust = np.exp(-((yy-(cy+0.25*(xx-cx)))**2)/(2*42**2))
        arr[...,0] -= dust*45; arr[...,1] -= dust*35; arr[...,2] -= dust*30

    elif kind == 'cartwheel_galaxy':
        arr = add_starfield(arr, seed=73, amount=2600)
        gx, gy = cx, cy
        r = np.sqrt((xx-gx)**2 + ((yy-gy)/0.82)**2)
        theta = np.arctan2((yy-gy)/0.82, xx-gx)
        ring = np.exp(-((r-210)**2)/(2*16**2))
        inner = np.exp(-((r-75)**2)/(2*20**2))
        spokes = (0.5 + 0.5*np.cos(10*theta + r/55)) * np.exp(-((r-145)**2)/(2*70**2))
        arr[...,0] += ring*170 + inner*180 + spokes*55
        arr[...,1] += ring*120 + inner*140 + spokes*50
        arr[...,2] += ring*255 + inner*90 + spokes*120
        # starburst knots
        for k in range(22):
            ang = 2*math.pi*k/22
            sx = gx + 210*math.cos(ang)
            sy = gy + 0.82*210*math.sin(ang)
            arr = draw_glow(arr, sx, sy, 28, (95,190,255), 0.35)
        arr = draw_glow(arr, gx, gy, 280, (180,130,255), 0.12, squash_y=0.82)

    else:
        arr = draw_glow(arr, cx, cy, 250, (255,255,255), 0.4)

    return np_to_img(arr).filter(ImageFilter.GaussianBlur(radius=0.4))


def render_frame(preset, frame_i):
    progress = frame_i / max(1, NFRAMES-1)
    clean = render_clean_scene(preset, progress)
    arr = np.array(clean).astype(np.float32)

    # live-stacking effect: noisy at first, clearer over time
    noise_amp = 110 * (1-progress)**1.55 + 5
    local = np.random.default_rng(1000 + frame_i)
    noise = local.normal(0, noise_amp, arr.shape)
    noisy = np.clip(arr + noise, 0, 255)
    # slight sharpening/reveal toward end
    img = Image.fromarray(noisy.astype(np.uint8), 'RGB')
    if progress > 0.55:
        img = img.filter(ImageFilter.SHARPEN)
    img = draw_hud(img, preset, frame_i, progress)
    return img


def safe_filename(name):
    return ''.join(c if c.isalnum() else '_' for c in name).strip('_')


def render_short(target_key):
    preset = PRESETS[target_key]
    out_mp4 = OUT_DIR / f"{safe_filename(preset['short_name'])}_educational_shorts_16s.mp4"
    out_png = OUT_DIR / f"{safe_filename(preset['short_name'])}_preview.png"
    print(f"Rendering {preset['display_name']} -> {out_mp4}")
    with imageio.get_writer(out_mp4, fps=FPS, codec='libx264', quality=8, macro_block_size=1) as writer:
        for i in range(NFRAMES):
            frame = render_frame(preset, i)
            if i == int(NFRAMES*0.86):
                frame.save(out_png)
            writer.append_data(np.array(frame))
    print('Saved:', out_mp4)
    print('Preview:', out_png)
    return out_mp4


if RENDER_ALL:
    for key in PRESETS:
        render_short(key)
else:
    render_short(TARGET)


Rendering Cartwheel Galaxy: Collision & Starburst Ring -> Cartwheel_Galaxy_educational_shorts_16s.mp4
Saved: Cartwheel_Galaxy_educational_shorts_16s.mp4
Preview: Cartwheel_Galaxy_preview.png
